In [1]:
using Pkg
Pkg.activate("C:/Users/ibzja/Documents/UPF_2022_2026/4t/2n_trimestre/Practiques_tutelades/CellBasedModels.jl")
using CellBasedModels 
using GeometryBasics
using Distributions
using GLMakie, Colors
Makie.inline!(true)
using CSV, DataFrames, Statistics
using Printf, JLD2
using SpecialFunctions
using LsqFit
using LinearAlgebra
using DifferentialEquations, StaticArrays

  Activating project at `C:\Users\ibzja\Documents\UPF_2022_2026\4t\2n_trimestre\Practiques_tutelades\CellBasedModels.jl`


In [ ]:
non_motile = ABM(2,
    agent = Dict(
        :vx => Float64,
        :vy => Float64,
        :v => Float64,  #Swimming speed
        :theta => Float64,
        :d => Float64,
        :l => Float64
    ),

    model = Dict(
        :D => Float64,
        :ve => Float64
    ),

    agentODE = quote    
        dt(x) = vx 
        dt(y) = vy     
        
    end,

    agentRule = quote  
        
        xmin, xmax = simBox[1,1], simBox[1,2]
        ymin, ymax = simBox[2,1], simBox[2,2]

        idx = Int(floor(Int, x/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)
        idy = Int(floor(Int, y/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)  
        if i1_ == 1
            for k in 1:10
                if rand() < p_add
                    @addAgent(
                        x = xmin + 10,
                        y = ymax/2 + 10 + rand()*(ymax - (ymax/2) - 10),
                        theta = 2 * pi,
                        l = 3
                    )
                end
            end
        end

        vx = ve[idx, idy] + sqrt(2*D*dt) * randn()
        vy = sqrt(2*D*dt) * randn()

        if x <= xmin
            x = xmin + (xmin - x)
            theta = pi - theta
        elseif x >= xmax
            @removeAgent()
        end

        if y >= ymax
            y = 2*ymax - y
            theta = 2*pi - theta
        end

        if y <= wall_y(x, 70, depth, 500, 20, 1000)
            x_new = clamp(round(Int, x/2), 1, 500)
            y_new = clamp(round(Int, y/2), 1, 500)

            if MYT[x_new, y_new] == 1 && MXR[x_new, y_new] == 1
                y = wall_y(x, 70, depth, 500, 20, 1000) + 1
                x = x + 1
                theta = theta + pi

            elseif MYT[x_new, y_new] == 1 && MXL[x_new, y_new] == 1
                y = wall_y(x, 70, depth, 500, 20, 1000) + 1
                x = x - 1
                theta = theta + pi

            elseif MYT[x_new, y_new] == 1 
                y = wall_y(x, 70, depth, 500, 20, 1000) + 1
                theta = 2*pi - theta

            elseif MXR[x_new, y_new] == 1
                x = x + 2
                theta = pi - theta

            elseif MXL[x_new, y_new] == 1
                x = x - 2
                theta = pi - theta

            elseif M0[x_new, y_new] == 1        # Esta dins, no pinta res allí
                t0 = round(Int, (t-1))
                x_old = x[t0]
                y_old = y[t0]
                nsteps = ceil(Int, max(abs(x - x_old), abs(y - y_old)))

                hit_type = nothing

                for s in 0:nsteps
                    xs = x_old + (x - x_old) * s / nsteps
                    ys = y_old + (y - y_old) * s / nsteps

                    xi = clamp(round(Int, xs/2), 1, 500)
                    yi = clamp(round(Int, ys/2), 1, 500)

                    if MXR[xi, yi]
                        hit_type = :right
                        break
                    elseif MXL[xi, yi]
                        hit_type = :left
                        break
                    elseif MYT[xi, yi]
                        hit_type = :horizontal
                        break
                    end
                end
                if hit_type == :right
                    x = x_old
                    theta = pi - theta

                elseif hit_type == :left
                    x = x_old
                    theta = pi - theta

                elseif hit_type == :horizontal
                    y = y_old
                    theta = 2*pi - theta
                end
                                
            end
        end
    end,

    agentAlg = CBMIntegrators.Heun(),
    mediumAlg=DifferentialEquations.Euler()
)

PARAMETERS
	x (Float64 agent)
	y (Float64 agent)
	xₘ (Float64 medium)
	yₘ (Float64 medium)
	Ds (Float64 agent)
	F (Float64 agent)
	active (Bool agent)
	methyl (Float64 agent)
	l (Float64 agent)
	S (Float64 agent)
	M (Float64 agent)
	d (Float64 agent)
	λ (Float64 agent)
	v (Float64 agent)
	A (Float64 agent)
	fx (Float64 agent)
	vx (Float64 agent)
	fy (Float64 agent)
	m (Float64 agent)
	Yp (Float64 agent)
	P (Float64 agent)
	pressure (Float64 agent)
	vy (Float64 agent)
	W (Float64 agent)
	G (Float64 agent)
	theta (Float64 agent)
	ε1 (Float64 model)
	α (Float64 model)
	Z (Float64 model)
	Dr_run (Float64 model)
	Ka (Float64 model)
	ε3 (Float64 model)
	ε0 (Float64 model)
	DMedium (Float64 model)
	delta (Float64 model)
	Ky (Float64 model)
	Kz (Float64 model)
	K (Float64 model)
	ε2 (Float64 model)
	Nrec (Float64 model)
	τm (Float64 model)
	Yy (Float64 model)
	Ki (Float64 model)
	ωFrec (Float64 model)
	mm (Float64 medium)
	ve (Float64 medium)


UPDATE RULES
mediumODE
 if @mediumInside()
    if

In [2]:
function wall_y(x, w, d, y0, xstart, xend)
    
    # outside wall region → flat at baseline
    if x < xstart || x > xend
        return y0
    end

    # shift x so pattern starts at xstart
    ξ = x - xstart

    # square-wave pattern (0 or -d), then shift up by y0
    return y0 - d * floor((1 + sign(sin(2π * ξ / w))) / 2)
end

wall_y (generic function with 1 method)

In [ ]:
        x1, x2, y1, y2 = 0, 1000, 0, 1000
        med1, med2 = 500, 500

        width = 62
        depth = 100
        y_start = y2/2
        x_start = 20
        x_end = x2
        dx = (x2 - x1) / med1
        dy = (y2 -y1) / med2

        xcoord(i1_) = x1 + (i1_ - 1) * dx
        ycoord(i2_) = y1 + (i2_ - 1) * dy
        Nx = med1
        Ny = med2

        mask_raw = zeros(Bool, med1, med2)
        M0 = zeros(Bool, med1, med2)
        MXL = zeros(Bool, med1, med2)
        MXR = zeros(Bool, med1, med2)
        MYT = zeros(Bool, med1, med2)

        for i in 1:Nx, j in 1:Ny
            x = xcoord(i)
            y = ycoord(j)
            mask_raw[i,j] = y < wall_y(x, width, depth, y_start, x_start, x_end)
        end

        for i in 2:Nx-1, j in 2:Ny-1
            if mask_raw[i,j] &&
            mask_raw[i+1,j] &&
            mask_raw[i-1,j] &&
            mask_raw[i,j+1] &&
            mask_raw[i,j-1] &&
            mask_raw[i+1,j+1] &&
            mask_raw[i+1,j-1] &&
            mask_raw[i-1,j+1] &&
            mask_raw[i-1,j-1]

                M0[i,j] = true
            end
        end

        for i in 2:Nx-1, j in 2:Ny-1

            if M0[i,j] == 1

                # --- HORIZONTAL LINES ---
                # bottom edge (neighbor below is outside)
                if M0[i, j-1] == 0
                    MYT[i,j] = true
                end

                # top edge (neighbor above is outside)
                if M0[i, j+1] == 0
                    MYT[i,j] = true
                end


                # --- LEFT WALL ---
                if M0[i-1, j] == 0
                    MXL[i,j] = true
                end


                # --- RIGHT WALL ---
                if M0[i+1, j] == 0
                    MXR[i,j] = true
                end

            end
        end

In [ ]:
Dc = 10
delta = 0.0025
depths = [100, 150, 200]
ves = [5, 10, 15]
# ns = [0.5, 1.0, 4.0, 6.0]

for (idx, ve) in enumerate(ves)

    for i in 1:length(depths)

        println("Running simulation with parameters: ", depths[i], " and ", n)

        x1, x2, y1, y2 = 0, 1000, 0, 1000
        med1, med2 = 500, 500

        width = 70
        depth = depths[i]
        y_start = y2/2
        x_start = 20
        x_end = x2
        dx = (x2 - x1) / med1
        dy = (y2 -y1) / med2

        xcoord(i1_) = x1 + (i1_ - 1) * dx
        ycoord(i2_) = y1 + (i2_ - 1) * dy
        Nx = med1
        Ny = med2

        mask_raw = zeros(Bool, med1, med2)
        M0 = zeros(Bool, med1, med2)
        MXL = zeros(Bool, med1, med2)
        MXR = zeros(Bool, med1, med2)
        MYT = zeros(Bool, med1, med2)

        for i in 1:Nx, j in 1:Ny
            x = xcoord(i)
            y = ycoord(j)
            mask_raw[i,j] = y < wall_y(x, width, depth, y_start, x_start, x_end)
        end

        for i in 2:Nx-1, j in 2:Ny-1
            if mask_raw[i,j] &&
            mask_raw[i+1,j] &&
            mask_raw[i-1,j] &&
            mask_raw[i,j+1] &&
            mask_raw[i,j-1] &&
            mask_raw[i+1,j+1] &&
            mask_raw[i+1,j-1] &&
            mask_raw[i-1,j+1] &&
            mask_raw[i-1,j-1]

                M0[i,j] = true
            end
        end

        for i in 2:Nx-1, j in 2:Ny-1

            if M0[i,j] == 1

                # --- HORIZONTAL LINES ---
                # bottom edge (neighbor below is outside)
                if M0[i, j-1] == 0
                    MYT[i,j] = true
                end

                # top edge (neighbor above is outside)
                if M0[i, j+1] == 0
                    MYT[i,j] = true
                end


                # --- LEFT WALL ---
                if M0[i-1, j] == 0
                    MXL[i,j] = true
                end


                # --- RIGHT WALL ---
                if M0[i+1, j] == 0
                    MXR[i,j] = true
                end

            end
        end

        com = Community(
            non_motile,
            N=500,
            dt=0.01,
            simBox = [x1 x2; y1 y2],
            NMedium = [med1, med2]
        )

        m = 1/100
        g = 1/10000
        d = 1

        com.D = 2 # ?

        com.v = 0    #Velocitat 0

        com.m = 1.        
        com.d = 1.        
        com.l = 3;

        com.x = zeros(com.N)
        com.y = zeros(com.N)

        count = 0

        while count < com.N
            x_try = rand(Uniform(1, 1000))   ## Limit 750 remember
            y_try = rand(Uniform(1, 750))

            # map to mask indices
            i = Int(floor(x_try / 2)) + 1
            j = Int(floor(y_try / 2)) + 1

            # bounds safety
            if i < 1 || i > y_start || j < 1 || j > y_start
                continue
            end

            if M0[i, j] == 0
                count += 1
                com.x[count] = x_try
                com.y[count] = y_try
            end
        end
        com.theta = rand(Uniform(0, 2*pi),com.N)  

        com.ve = zeros(com.NMedium[1], com.NMedium[2])

        for i1 in 1:com.NMedium[1], i2 in 1:com.NMedium[2]
            y = ycoord(i2)

            if y <= y_start
                # Inside crypt → no flow
                com.ve[i1, i2] = 0.0

            elseif y < y_start + L
                # Transition region → smooth increase
                f = (y - y_start) / L
                com.ve[i1, i2] = ve * f^2

            else
                # Bulk lumen → full flow
                com.ve[i1, i2] = ve
            end
        end

        outfile = "nm_ve$(ve)_lc_$(depths[i]).jld2"
        steps = 150000

        loadToPlatform!(com, preallocateAgents=100000)

        jldopen(outfile, "w") do file
            
            for step in 1:steps
                CellBasedModels.step!(com)
                if step % 1000 == 0
                    stepname = @sprintf("step_%06d", step)
                    g = JLD2.Group(file, stepname)

                    # Agent-level arrays (length = N)
                    g["x"] = copy(com.x)
                    g["y"] = copy(com.y)
                    g["theta"] = copy(com.theta)

                end
            end
        end
    end
end


Running simulation with parameters: 100 and 0.5
Running simulation with parameters: 150 and 0.5
Running simulation with parameters: 200 and 0.5
Running simulation with parameters: 100 and 1.0
Running simulation with parameters: 150 and 1.0
Running simulation with parameters: 200 and 1.0
Running simulation with parameters: 100 and 4.0
Running simulation with parameters: 150 and 4.0
Running simulation with parameters: 200 and 4.0
Running simulation with parameters: 100 and 6.0
Running simulation with parameters: 150 and 6.0
Running simulation with parameters: 200 and 6.0


In [34]:
steps = 100000
steps_1 = 100:100:steps
sizes = length(steps_1)

1000

In [14]:
Dc = 10
delta = 0.0025
depths = [100, 150, 200]
ve = 10
ns = [0.5, 1.0, 4.0, 6.0]
steps = 100000

for (idx, n) in enumerate(ns)

    for i in 1:length(depths)

        println("Running simulation with parameters: ", depths[i], " and ", n)

        x1, x2, y1, y2 = 0, 500, 0, 500
        med1, med2 = 250, 250

        width = 62
        depth = depths[i]
        y_start = y2/2
        x_start = 20
        x_end = x2
        dx = (x2 - x1) / med1
        dy = (y2 -y1) / med2

        xcoord(i1_) = x1 + (i1_ - 1) * dx
        ycoord(i2_) = y1 + (i2_ - 1) * dy
        Nx = med1
        Ny = med2

        mask_raw = zeros(Bool, med1, med2)
        M0 = zeros(Bool, med1, med2)
        MXL = zeros(Bool, med1, med2)
        MXR = zeros(Bool, med1, med2)
        MYT = zeros(Bool, med1, med2)

        for i in 1:Nx, j in 1:Ny
            x = xcoord(i)
            y = ycoord(j)
            mask_raw[i,j] = y < wall_y(x, width, depth, y_start, x_start, x_end)
        end

        for i in 2:Nx-1, j in 2:Ny-1
            if mask_raw[i,j] &&
            mask_raw[i+1,j] &&
            mask_raw[i-1,j] &&
            mask_raw[i,j+1] &&
            mask_raw[i,j-1] &&
            mask_raw[i+1,j+1] &&
            mask_raw[i+1,j-1] &&
            mask_raw[i-1,j+1] &&
            mask_raw[i-1,j-1]

                M0[i,j] = true
            end
        end

        for i in 2:Nx-1, j in 2:Ny-1

            if M0[i,j] == 1

                # --- HORIZONTAL LINES ---
                # bottom edge (neighbor below is outside)
                if M0[i, j-1] == 0
                    MYT[i,j] = true
                end

                # top edge (neighbor above is outside)
                if M0[i, j+1] == 0
                    MYT[i,j] = true
                end


                # --- LEFT WALL ---
                if M0[i-1, j] == 0
                    MXL[i,j] = true
                end


                # --- RIGHT WALL ---
                if M0[i+1, j] == 0
                    MXR[i,j] = true
                end

            end
        end

        data = Dict{Int, Any}()
        jldopen("c_n$(n)_lc_$(depths[i]).jld2", "r") do file
            for step in 100:100:steps
                key = file[@sprintf("step_%06d", step)]
                data[step] = Dict(
                    "x" => copy(key["x"]),
                    "y" => copy(key["y"]),
                    "mm_grid" => copy(key["mm_grid"])
                )
            end
        end

        # Fer matrius per cada set
        ncripts = 8
        right_wall = findall(MXR[:, 100] .== 1) .* 2
        left_wall = findall(MXL[:, 100] .== 1) .* 2
        x1_1, x1_2, x1_3, x1_4, x1_5, x1_6, x1_7, x1_8 = right_wall[1:8]
        x2_1, x2_2, x2_3, x2_4, x2_5, x2_6, x2_7, x2_8 = left_wall[2:9]

        y_bottom = y_start - depth
        y_top = y_start

        steps_1 = 100:100:steps
        sizes = length(steps_1)
        counts = zeros(ncripts, sizes)

        for i in 1:sizes
            step = steps_1[i]
            g = data[step]
            x = g["x"]
            y = g["y"]

            count1 = 0
            count2 = 0
            count3 = 0
            count4 = 0
            count5 = 0
            count6 = 0
            count7 = 0
            count8 = 0

            for j in 1:length(x)
            
                if x2_1 > x[j] > x1_1 && y[j] < y_top
                    count1 += 1
                elseif x2_2 > x[j] > x1_2 && y[j] < y_top
                    count2 += 1
                elseif x2_3 > x[j] > x1_3 && y[j] < y_top
                    count3 += 1
                elseif x2_4 > x[j] > x1_4 && y[j] < y_top
                    count4 += 1
                elseif x2_5 > x[j] > x1_5 && y[j] < y_top
                    count5 += 1
                elseif x2_6 > x[j] > x1_6 && y[j] < y_top
                    count6 += 1
                elseif x2_7 > x[j] > x1_7 && y[j] < y_top
                    count7 += 1
                elseif x2_8 > x[j] > x1_8 && y[j] < y_top
                    count8 += 1
                end
            end

            counts[1, i] = count1
            counts[2, i] = count2
            counts[3, i] = count3
            counts[4, i] = count4
            counts[5, i] = count5
            counts[6, i] = count6
            counts[7, i] = count7
            counts[8, i] = count8
        end

        fig = Figure(size=(900, 600))
        ax = Axis(fig[1, 1], xlabel = "Time", ylabel = "Nº cells", title = "Cript colonization over time")
        lines!(ax, (1:sizes), counts[1, :], label = "Cript 1", linewidth = 1)
        lines!(ax, (1:sizes), counts[2, :], label = "Cript 2", linewidth = 1)
        lines!(ax, (1:sizes), counts[3, :], label = "Cript 3", linewidth = 1)
        lines!(ax, (1:sizes), counts[4, :], label = "Cript 4", linewidth = 1)
        lines!(ax, (1:sizes), counts[5, :], label = "Cript 5", linewidth = 1)
        lines!(ax, (1:sizes), counts[6, :], label = "Cript 6", linewidth = 1)
        lines!(ax, (1:sizes), counts[7, :], label = "Cript 7", linewidth = 1)
        lines!(ax, (1:sizes), counts[8, :], label = "Cript 8", linewidth = 1)

        fig[1,2] = Legend(fig, ax, "Cripts")

        save("Plots/Cripts/Test_sims/colonization_time_n$(n)_lc$(depths[i]).png", fig)


        grid_values = zeros(ncripts, sizes)

        x2 = Int.(left_wall[2:9] ./ 2)
        x1 = Int.(right_wall[1:8] ./2)
        y1 = Int(y_bottom / 2)
        y2 = Int(y_start / 2)

        for i in 1:sizes
            step = steps_1[i]
            g = data[step]
            grid = g["mm_grid"]
            for j in 1:ncripts
            
                cript = grid[x1[j]:x2[j], y1:y2]
                mm_mean = mean(cript)

                grid_values[j, i] = mm_mean

            end
        end

        fig2 = Figure(size=(900, 600))
        ax2 = Axis(fig2[1, 1], xlabel = "Time", ylabel = "Nº cells", title = "Cript mm concentration over time")
        lines!(ax2, (1:sizes), grid_values[1, :], label = "Cript 1")
        lines!(ax2, (1:sizes), grid_values[2, :], label = "Cript 2")
        lines!(ax2, (1:sizes), grid_values[3, :], label = "Cript 3")
        lines!(ax2, (1:sizes), grid_values[4, :], label = "Cript 4")
        lines!(ax2, (1:sizes), grid_values[5, :], label = "Cript 5")
        lines!(ax2, (1:sizes), grid_values[6, :], label = "Cript 6")
        lines!(ax2, (1:sizes), grid_values[7, :], label = "Cript 7")
        lines!(ax2, (1:sizes), grid_values[8, :], label = "Cript 8")
        fig2[1,2] = Legend(fig2, ax2, "Cripts")

        save("Plots/Cripts/Test_sims/concentration_time_n$(n)_lc$(depths[i]).png", fig2)
    end
end

Running simulation with parameters: 100 and 0.5
Running simulation with parameters: 150 and 0.5
Running simulation with parameters: 200 and 0.5
Running simulation with parameters: 100 and 1.0
Running simulation with parameters: 150 and 1.0
Running simulation with parameters: 200 and 1.0
Running simulation with parameters: 100 and 4.0
Running simulation with parameters: 150 and 4.0
Running simulation with parameters: 200 and 4.0
Running simulation with parameters: 100 and 6.0
Running simulation with parameters: 150 and 6.0
Running simulation with parameters: 200 and 6.0


# Non-chemotactic

In [ ]:
non_chemo = ABM(2,
    agent = Dict(
        :vx => Float64,
        :vy => Float64,
        :v => Float64,  #Swimming speed
        :theta => Float64,
        :d => Float64,
        :l => Float64,
        :m => Float64,
        :active => Bool

    ),

    model = Dict(

        :Dr_run => Float64,
        :ωFrec => Float64     #Basal switching frequency

    ),

    agentODE = quote  

        dt(x) = vx 
        dt(y) = vy     
        
    end,

    agentRule = quote

        xmin, xmax = simBox[1,1], simBox[1,2]
        ymin, ymax = simBox[2,1], simBox[2,2]

        idx = Int(floor(Int, x/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)
        idy = Int(floor(Int, y/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)        
       
     # Adding population every 100 steps
        p_add = 0.01

        if i1_ == 1
            for k in 1:10
                if rand() < p_add
                    @addAgent(
                        x = xmin + 10,
                        y = ymax/2 + 10 + rand()*(ymax - (ymax/2) - 10),
                        theta = 2 * pi,
                        l = 3
                    )
                end
            end
        end

        v_run = v
        v_tumble = 0.25 
        speed = active ? v_run : v_tumble

        Dr_tumble = 6.2      
        Dr_total = active ? Dr_run : Dr_tumble

        λ = ωFrec
        P = 1 - exp(-λ * dt)

        if active 
            P_rand = rand() 
                                                  
            if P_rand < P            
                active = false
                vx = speed* cos(theta) + ve[idx, idy]
                vy = speed* sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn() 
                        
            else    
                active = true
                vx = speed * cos(theta) + ve[idx, idy]
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()
                     
            end

        else
            P_rand = rand()

            if P_rand < P
                active = true
                vx = speed * cos(theta) + ve[idx, idy]
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()

            else
                active = false
                vx = speed* cos(theta) + ve[idx, idy]
                vy = speed* sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()

            end
        end

        if x <= xmin
            x = xmin + (xmin - x)
            theta = pi - theta
        elseif x >= xmax
            @removeAgent()
        end

        if y >= 750
            y = 2*750 - y
            theta = 2*pi - theta
        end

        if y <= wall_y(x, 70, 200, 500, 20, 1000)
            x_new = clamp(round(Int, x/2), 1, 500)
            y_new = clamp(round(Int, y/2), 1, 500)

            if MYT[x_new, y_new] == 1 && MXR[x_new, y_new] == 1
                y = wall_y(x, 70, 200, 500, 20, 1000) + 1
                x = x + 1
                theta = theta + pi

            elseif MYT[x_new, y_new] == 1 && MXL[x_new, y_new] == 1
                y = wall_y(x, 70, 200, 500, 20, 1000) + 1
                x = x - 1
                theta = theta + pi

            elseif MYT[x_new, y_new] == 1 
                y = wall_y(x, 70, 200, 500, 20, 1000) + 1
                theta = 2*pi - theta

            elseif MXR[x_new, y_new] == 1
                x = x + 2
                theta = pi - theta

            elseif MXL[x_new, y_new] == 1
                x = x - 2
                theta = pi - theta

            elseif M0[x_new, y_new] == 1        # Esta dins, no pinta res allí
                t0 = round(Int, (t-1))
                x_old = x[t0]
                y_old = y[t0]
                nsteps = ceil(Int, max(abs(x - x_old), abs(y - y_old)))

                hit_type = nothing

                for s in 0:nsteps
                    xs = x_old + (x - x_old) * s / nsteps
                    ys = y_old + (y - y_old) * s / nsteps

                    xi = clamp(round(Int, xs/2), 1, 500)
                    yi = clamp(round(Int, ys/2), 1, 500)

                    if MXR[xi, yi]
                        hit_type = :right
                        break
                    elseif MXL[xi, yi]
                        hit_type = :left
                        break
                    elseif MYT[xi, yi]
                        hit_type = :horizontal
                        break
                    end
                end
                if hit_type == :right
                    x = x_old
                    theta = pi - theta

                elseif hit_type == :left
                    x = x_old
                    theta = pi - theta

                elseif hit_type == :horizontal
                    y = y_old
                    theta = 2*pi - theta
                end
                                
            end
        end
    end,
    agentAlg = CBMIntegrators.Heun(),
    mediumAlg=DifferentialEquations.Euler()
)

In [ ]:
depths = [100, 150, 200]
ves = [5, 10, 15]

for (idx, ve) in enumerate(ves)

    for i in 1:length(depths)

        println("Running simulation with parameters: ", depths[i], " and ", n)

        x1, x2, y1, y2 = 0, 1000, 0, 1000
        med1, med2 = 500, 500

        width = 70
        depth = depths[i]
        y_start = y2/2
        x_start = 20
        x_end = x2
        dx = (x2 - x1) / med1
        dy = (y2 -y1) / med2

        xcoord(i1_) = x1 + (i1_ - 1) * dx
        ycoord(i2_) = y1 + (i2_ - 1) * dy
        Nx = med1
        Ny = med2

        mask_raw = zeros(Bool, med1, med2)
        M0 = zeros(Bool, med1, med2)
        MXL = zeros(Bool, med1, med2)
        MXR = zeros(Bool, med1, med2)
        MYT = zeros(Bool, med1, med2)

        for i in 1:Nx, j in 1:Ny
            x = xcoord(i)
            y = ycoord(j)
            mask_raw[i,j] = y < wall_y(x, width, depth, y_start, x_start, x_end)
        end

        for i in 2:Nx-1, j in 2:Ny-1
            if mask_raw[i,j] &&
            mask_raw[i+1,j] &&
            mask_raw[i-1,j] &&
            mask_raw[i,j+1] &&
            mask_raw[i,j-1] &&
            mask_raw[i+1,j+1] &&
            mask_raw[i+1,j-1] &&
            mask_raw[i-1,j+1] &&
            mask_raw[i-1,j-1]

                M0[i,j] = true
            end
        end

        for i in 2:Nx-1, j in 2:Ny-1

            if M0[i,j] == 1

                # --- HORIZONTAL LINES ---
                # bottom edge (neighbor below is outside)
                if M0[i, j-1] == 0
                    MYT[i,j] = true
                end

                # top edge (neighbor above is outside)
                if M0[i, j+1] == 0
                    MYT[i,j] = true
                end


                # --- LEFT WALL ---
                if M0[i-1, j] == 0
                    MXL[i,j] = true
                end


                # --- RIGHT WALL ---
                if M0[i+1, j] == 0
                    MXR[i,j] = true
                end

            end
        end

        com = Community(
            non_motile,
            N=500,
            dt=0.01,
            simBox = [x1 x2; y1 y2],
            NMedium = [med1, med2]
        )

        m = 1/100
        g = 1/10000
        d = 1

        com.Dr_run = 0.0062

        com.v = 20    #Velocitat 0

        com.m = 1.        
        com.d = 1.        
        com.l = 3;

        com.x = zeros(com.N)
        com.y = zeros(com.N)

        count = 0

        while count < com.N
            x_try = rand(Uniform(1, 1000))   ## Limit 750 remember
            y_try = rand(Uniform(1, 750))

            # map to mask indices
            i = Int(floor(x_try / 2)) + 1
            j = Int(floor(y_try / 2)) + 1

            # bounds safety
            if i < 1 || i > y_start || j < 1 || j > y_start
                continue
            end

            if M0[i, j] == 0
                count += 1
                com.x[count] = x_try
                com.y[count] = y_try
            end
        end
        com.theta = rand(Uniform(0, 2*pi),com.N)  

        com.ve = zeros(com.NMedium[1], com.NMedium[2])
        com.ωFrec = 1.3

        for i1 in 1:com.NMedium[1], i2 in 1:com.NMedium[2]
            y = ycoord(i2)

            if y <= y_start
                # Inside crypt → no flow
                com.ve[i1, i2] = 0.0

            elseif y < y_start + L
                # Transition region → smooth increase
                f = (y - y_start) / L
                com.ve[i1, i2] = ve * f^2

            else
                # Bulk lumen → full flow
                com.ve[i1, i2] = ve
            end
        end

        outfile = "nm_ve$(ve)_lc_$(depths[i]).jld2"
        steps = 150000

        loadToPlatform!(com, preallocateAgents=100000)

        jldopen(outfile, "w") do file
            
            for step in 1:steps
                CellBasedModels.step!(com)
                if step % 1000 == 0
                    stepname = @sprintf("step_%06d", step)
                    g = JLD2.Group(file, stepname)

                    # Agent-level arrays (length = N)
                    g["x"] = copy(com.x)
                    g["y"] = copy(com.y)
                    g["theta"] = copy(com.theta)

                end
            end
        end
    end
end
